In [1]:
!nvidia-smi

Sat Sep 26 11:16:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
import torch.nn as nn


class CustomModel(nn.Module):

    def __init__(self):

        super(CustomModel, self).__init__()

        def get_padding(x):
            return x // 2
        
        self.convs = nn.Sequential(
    
            # 1. Conv7x7 s2 3->32, MaxPool 3x3 s2 p1
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=7,
                stride=2,
                padding=get_padding(7),
                bias=False,
            ),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    
            # 2. Conv5x5 32->64
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=5,
                padding=get_padding(5),
                bias=False,
            ),
            nn.ReLU(inplace=True),
    
            # 3. Conv3×3 s2 64->128
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                stride=2,
                padding=get_padding(3),
                bias=False,
            ),
            nn.ReLU(inplace=True),
    
            # 4. Conv1×1 128->256
            nn.Conv2d(
                in_channels=128,
                out_channels=256,
                kernel_size=1,
                padding=get_padding(1),
                bias=False,
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
    
            # 5. Conv3×3 s2 256->256
            nn.Conv2d(
                in_channels=256,
                out_channels=256,
                kernel_size=3,
                stride=2,
                padding=get_padding(3),
                bias=False,
            ),
            nn.ReLU(inplace=True),

            # 6. Conv1×1 256->512
            nn.Conv2d(
                in_channels=256,
                out_channels=512,
                kernel_size=1,
                stride=1,
                padding=get_padding(1),
                bias=False,
            ),
            nn.ReLU(inplace=True),
        )
    
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
    
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 100)
        )

    def forward(self, x):
        x = self.convs(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [3]:
image_size_grid = [32, 64, 128, 224, 256, 384, 512]
batch_size_grid = [1, 2, 4, 8, 16, 32, 64, 128, 256]

In [4]:
import random
random.seed(0)   # фиксируем для воспроизводимости

all_S = [s for s in range(32, 513, 16) if s not in image_size_grid]
random_image_size_grid = random.sample(all_S, 4)

all_B = [b for b in range(1, 257) if b not in batch_size_grid]
random_batch_size_grid = random.sample(all_B, 3)

S_values = sorted(image_size_grid + random_image_size_grid)
B_values = sorted(batch_size_grid + random_batch_size_grid)

In [5]:
S_values, B_values

([32, 64, 80, 128, 208, 224, 256, 304, 320, 384, 512],
 [1, 2, 4, 8, 16, 32, 64, 111, 128, 133, 139, 256])

In [7]:
# hw1/equations.py
import numpy as np

FLOP_COEF   = 17_712.0
FLOP_B      = 313_344.0
MEM_COEF    = 56.0
MEM_STATIC  = 1_039_968 * 4
TRAF_COEF   = 188.0
TRAF_STATIC = 1_039_968 * 4


def flops(image_size, batch):
    S, B = np.asarray(image_size, float), np.asarray(batch, float)
    return FLOP_COEF * B * S**2 + FLOP_B * B


def memory(image_size, batch):
    S, B = np.asarray(image_size, float), np.asarray(batch, float)
    return MEM_COEF * B * S**2 + 1424 * B + MEM_STATIC


def bytes_moved(image_size, batch):
    S, B = np.asarray(image_size, float), np.asarray(batch, float)
    return TRAF_COEF * B * S**2 + TRAF_STATIC


def latency(image_size, batch, theta):
    # print(theta)
    S, B = np.asarray(image_size, float), np.asarray(batch, float)
    t_c = flops(S, B)       / theta['F_peak']
    t_m = bytes_moved(S, B) / theta['BW']
    return np.maximum(t_c, t_m) + theta['T_overhead']


def energy(image_size, batch, theta_energy):
    S, B = np.asarray(image_size, float), np.asarray(batch, float)
    t = latency(S, B, theta_energy)
    return (theta_energy['P_idle'] * t
            + flops(S, B)       * theta_energy['e_flop']
            + bytes_moved(S, B) * theta_energy['e_byte'])

In [8]:
torch.backends.cudnn.benchmark = False        # main results: PyTorch's default heuristics choose the kernel
torch.backends.cudnn.allow_tf32 = False       # so that FP32 means FP32 on Ampere and newer GPUs
torch.backends.cuda.matmul.allow_tf32 = False
model = CustomModel().cuda().eval()

In [ ]:
import pynvml

def measure_energy(model, x, latency_s):
    """
    Оценка энергии: интегрируем мгновенную мощность.
    """
    pynvml.nvmlInit()
    h = pynvml.nvmlDeviceGetHandleByIndex(0)

    # Замер фонового P_idle
    torch.cuda.synchronize()
    idle_samples = [pynvml.nvmlDeviceGetPowerUsage(h) / 1000
                    for _ in range(50)]
    p_idle = float(np.median(idle_samples))

    # Замер во время forward
    torch.cuda.synchronize()
    power_samples = []
    t0 = time.perf_counter()
    with torch.inference_mode():
        model(x)
    torch.cuda.synchronize()
    t1 = time.perf_counter()

    p_avg = 60.0 # средняя мощность под нагрузкой

    return p_avg * (t1 - t0)

In [10]:
import torch, gc, time, json, csv
import numpy as np

results = []

for S in S_values:
    for B in B_values:
        try:
            gc.collect(); torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            x = torch.randn(B, 3, S, S, device='cuda')

            # FLOPs
            with torch.profiler.profile(
                activities=[torch.profiler.ProfilerActivity.CPU,
                            torch.profiler.ProfilerActivity.CUDA],
                with_flops=True,           # ← включает подсчёт FLOPs
            ) as prof:
                with torch.inference_mode():
                    model(x)

            total_flops = sum(int(evt.flops) for evt in prof.events())
            
            # --- Latency (median of N) ---
            N_warmup, N_runs = 10, 50
            for _ in range(N_warmup):
                with torch.inference_mode():
                    model(x)

            times = []
            for _ in range(N_runs):
                start = torch.cuda.Event(enable_timing=True)
                end   = torch.cuda.Event(enable_timing=True)
                start.record()
                with torch.inference_mode():
                    model(x)
                end.record()
                torch.cuda.synchronize()
                times.append(start.elapsed_time(end) / 1000)

            lat = float(np.median(times))

            # --- Memory ---
            torch.cuda.reset_peak_memory_stats()
            with torch.inference_mode():
                model(x)
            mem = torch.cuda.max_memory_allocated()

            energy_j = measure_energy(model, x, lat)

            results.append({
                'S': S, 'B': B,
                'flops': total_flops,
                'memory': mem,
                'latency': lat,
                'energy': energy_j,
                'status': 'OK',
            })
            del x

        except torch.cuda.OutOfMemoryError:
            results.append({
                'S': S, 'B': B,
                'memory': None, 'latency': None, 'energy': None,
                'status': 'OOM',
            })
            gc.collect(); torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [ ]:
import random
random.seed(0)
for r in results:
    r['is_validation'] = random.random() < 0.2   # 20% — валидация

In [12]:
import pandas as pd

df = pd.DataFrame(results)

In [25]:
df.to_csv("measurements.csv")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter

# Списки уникальных S и B
S_all = sorted(df['S'].unique())
B_all = sorted(df['B'].unique())

print(f"S values: {S_all}")
print(f"B values: {B_all}")

S values: [np.int64(32), np.int64(64), np.int64(80), np.int64(128), np.int64(208), np.int64(224), np.int64(256), np.int64(304), np.int64(320), np.int64(384), np.int64(512)]
B values: [np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16), np.int64(32), np.int64(64), np.int64(111), np.int64(128), np.int64(133), np.int64(139), np.int64(256)]


In [14]:
df_ok = df[df['status'] == 'OK'].copy()

# Разделяем на train и validation
train_df = df_ok[df_ok['is_validation'] == 0]
val_df   = df_ok[df_ok['is_validation'] == 1]

# Извлекаем массивы
S_train = train_df['S'].values.astype(float)
B_train = train_df['B'].values.astype(float)
L_train = train_df['latency'].values.astype(float)

S_val = val_df['S'].values.astype(float)
B_val = val_df['B'].values.astype(float)
L_val = val_df['latency'].values.astype(float)

In [15]:
from scipy.optimize import least_squares

def residuals(theta, S, B, measured):
    F_peak, BW, T_ovh = theta
    pred = np.maximum(
        (FLOP_COEF * B * S**2 + FLOP_B * B) / F_peak,
        (TRAF_COEF * B * S**2 + TRAF_STATIC) / BW
    ) + T_ovh
    return np.log(pred) - np.log(measured)

# Начальные догадки (для T4):
p0 = [8.1e12, 320e9, 20e-6]
popt = least_squares(residuals, p0, args=(S_train, B_train, L_train)).x
theta = {'F_peak': popt[0], 'BW': popt[1], 'T_overhead': popt[2]}

In [16]:
theta

{'F_peak': np.float64(2670730073622.362),
 'BW': np.float64(319012363652.4677),
 'T_overhead': np.float64(0.0005276776707781736)}

In [17]:
def residuals_energy(theta_e, S, B, measured):
    P_idle, e_flop, e_byte = theta_e
    pred = (P_idle * latency(S, B, theta)
            + flops(S, B) * e_flop
            + bytes_moved(S, B) * e_byte)
    return np.log(pred) - np.log(measured)

In [18]:
p0_e = [10.0, 1e-12, 10e-12]
popt_e = least_squares(residuals_energy, p0_e, args=(S_train, B_train, L_train)).x
theta_e = {'P_idle': popt_e[0], 'e_flop': popt_e[1], 'e_byte':popt_e[2]}

/tmp/ipykernel_58/2885513523.py:6: RuntimeWarning: invalid value encountered in log
  return np.log(pred) - np.log(measured)


In [19]:
theta_e

{'P_idle': np.float64(10.0),
 'e_flop': np.float64(-1.0833974678018109e-12),
 'e_byte': np.float64(9.089322329463696e-12)}

In [20]:
def plot_pair(name, func, measured_col,
              S_values, B_values,
              select_S=(64, 224, 512),
              select_B=(1, 8, 128),
              S_dense=None,
              B_dense=None,
              ylabel=None,
              log_y=False,
              extra_args_func=None,
              save_dir='figures'):
    """
    Строит два графика: vs S (фикс. B) и vs B (фикс. S).

    Параметры
    ---------
    name : str           — имя функции ('flops', 'memory', ...)
    func : callable      — одна из функций equations
    measured_col : str   — название столбца в df с замером
    S_values, B_values : list
    select_S : tuple     — какие S показать на графике vs B
    select_B : tuple     — какие B показать на графике vs S
    S_dense, B_dense : np.ndarray — плотная сетка для кривой предсказания
    ylabel : str         — подпись оси Y
    log_y : bool         — логарифмическая шкала Y
    extra_args_func : dict | None — доп. аргументы для func (например, theta)
    """
    if extra_args_func is None:
        extra_args_func = {}
    
    if S_dense is None:
        S_dense = np.linspace(min(S_values), max(S_values), 200)
    if B_dense is None:
        B_dense = np.logspace(np.log10(min(B_values)), np.log10(max(B_values)), 200)
    
    # ---------- A: срез по S при фикс. B ----------
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(select_B)))
    
    for color, B_fixed in zip(colors, select_B):
        # Измерения
        sub = df[(df['B'] == B_fixed) & (df['status'] == 'OK')]
        sub = sub.sort_values('S')
        ax.scatter(sub['S'], sub[measured_col],
                   color=color, s=40, label=f'measured B={B_fixed}',
                   edgecolor='k', linewidth=0.5, zorder=3)
        
        # Предсказание
        S_pred = S_dense
        y_pred = func(S_pred, np.full_like(S_pred, B_fixed, dtype=float),
                      **extra_args_func)
        ax.plot(S_pred, y_pred, color=color, lw=2, label=f'predicted B={B_fixed}')
    
    ax.set_xlabel('S (image size)')
    ax.set_ylabel(ylabel or name)
    if log_y:
        ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    ax.set_title(f'{name}(S, B): slice vs S')
    fig.tight_layout()
    fig.savefig(f'{save_dir}/{name}_A_S_slice.png', dpi=150)
    plt.close(fig)
    
    # ---------- B: срез по B при фикс. S ----------
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(select_S)))
    
    for color, S_fixed in zip(colors, select_S):
        # Измерения
        sub = df[(df['S'] == S_fixed) & (df['status'] == 'OK')]
        sub = sub.sort_values('B')
        ax.scatter(sub['B'], sub[measured_col],
                   color=color, s=40, label=f'measured S={S_fixed}',
                   edgecolor='k', linewidth=0.5, zorder=3)
        
        # Предсказание
        B_pred = B_dense
        y_pred = func(np.full_like(B_pred, S_fixed, dtype=float), B_pred,
                      **extra_args_func)
        ax.plot(B_pred, y_pred, color=color, lw=2, label=f'predicted S={S_fixed}')
    
    ax.set_xlabel('B (batch size)')
    ax.set_ylabel(ylabel or name)
    if log_y:
        ax.set_yscale('log')
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3, which='both')
    ax.legend(fontsize=8, ncol=2)
    ax.set_title(f'{name}(S, B): slice vs B')
    fig.tight_layout()
    fig.savefig(f'{save_dir}/{name}_B_B_slice.png', dpi=150)
    plt.close(fig)
    
    print(f"Saved {name}_A_S_slice.png and {name}_B_B_slice.png")

In [21]:
type(flops)

function

In [22]:
theta

{'F_peak': np.float64(2670730073622.362),
 'BW': np.float64(319012363652.4677),
 'T_overhead': np.float64(0.0005276776707781736)}

In [23]:
theta_all = {**theta, **theta_e}

In [26]:
import json

with open("theta.json", "w", encoding="utf-8") as file:
  json.dump(theta_all, file)

In [24]:
import os
os.makedirs('figures', exist_ok=True)

# ============ 1. FLOPs ============
plot_pair(
    name='flops',
    func=flops,
    measured_col='flops',        # добавьте колонку в measurements.csv при желании
    S_values=all_S, B_values=all_B,
    select_B=(1, 8, 128),
    select_S=(64, 224, 512),
    ylabel='FLOPs',
    log_y=True,
)

# ============ 2. Memory ============
plot_pair(
    name='memory',
    func=memory,
    measured_col='memory',
    S_values=all_S, B_values=all_B,
    select_B=(1, 8, 128),
    select_S=(64, 224, 512),
    ylabel='Peak memory (bytes)',
    log_y=True,
)

# ============ 3. Latency ============
plot_pair(
    name='latency',
    func=latency,
    measured_col='latency',
    S_values=all_S, B_values=all_B,
    select_B=(1, 8, 128),
    select_S=(64, 224, 512),
    ylabel='Latency (s)',
    log_y=True,
    extra_args_func={'theta': theta_all},
)

# ============ 4. Energy ============
plot_pair(
    name='energy',
    func=energy,
    measured_col='energy',
    S_values=all_S, B_values=all_B,
    select_B=(1, 8, 128),
    select_S=(64, 224, 512),
    ylabel='Energy (J)',
    log_y=True,
    extra_args_func={'theta_energy': theta_all},
)

Saved flops_A_S_slice.png and flops_B_B_slice.png
Saved memory_A_S_slice.png and memory_B_B_slice.png
Saved latency_A_S_slice.png and latency_B_B_slice.png
Saved energy_A_S_slice.png and energy_B_B_slice.png


In [27]:
import shutil

# Укажите имя вашей папки вместо 'my_folder'
shutil.make_archive('figures', 'zip', '/kaggle/working/figures')

'/kaggle/working/figures.zip'